# IMPORT

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [29]:
raw_data = pd.read_csv('employee_attrition_2026_master.csv')
dict01 = pd.read_csv('data_dictionary.csv')

# Checking Missing / Duplicate / raw_data Type

In [30]:
raw_data.head()

,employee_id,age,generation,gender,department,job_level,is_manager,tenure_years,salary_usd,salary_vs_market_pct,...,burnout_score,manager_relationship_score,manager_1on1_per_month,engagement_score,promotions_last_2y,career_growth_score,recent_layoff_round,team_size,attrition_reason,attrition
0,EMP000000,26,Gen Z,Female,Marketing,Junior,0,5.5,70000,5.2,...,3.6,2,0,62,0,2,0,22,stayed,0
1,EMP000001,35,Millennial,Female,Sales,Junior,0,0.5,64000,-15.5,...,4.8,3,2,79,1,2,0,24,stayed,0
2,EMP000002,23,Gen Z,Male,HR,Senior,0,2.0,142000,-15.9,...,10.0,1,2,43,0,4,1,3,poor_management,1
3,EMP000003,20,Gen Z,Female,Sales,Manager,1,2.2,147000,2.6,...,4.3,1,4,64,1,3,0,24,stayed,0
4,EMP000004,40,Millennial,Female,Data,Senior,0,6.0,141000,11.3,...,2.9,1,1,92,0,2,0,5,stayed,0


In [31]:
dict01

,column,type,description
0,employee_id,string,Unique employee id
1,age,int,Age in years
2,generation,string,Gen Z / Millennial / Gen X / Boomer (derived f...
3,gender,string,Male / Female / Other
4,department,string,"Department (Engineering, Product, Sales, Opera..."
5,job_level,string,Junior / Mid / Senior / Lead / Manager / Director
6,is_manager,int,1 if a people-manager role (Lead/Manager/Direc...
7,tenure_years,float,Years at the company
8,salary_usd,int,Annual salary (USD)
9,salary_vs_market_pct,float,"Salary vs market benchmark (%), negative = und..."


In [32]:
print(raw_data.dtypes)

employee_id                    object
age                             int64
generation                     object
gender                         object
department                     object
job_level                      object
is_manager                      int64
tenure_years                  float64
salary_usd                      int64
salary_vs_market_pct          float64
work_arrangement               object
rto_mandate                     int64
days_in_office_required         int64
commute_minutes                 int64
prefers_remote                  int64
flexibility_importance          int64
ai_tools_adoption              object
weekly_hours                    int64
after_hours_work                int64
burnout_score                 float64
manager_relationship_score      int64
manager_1on1_per_month          int64
engagement_score                int64
promotions_last_2y              int64
career_growth_score             int64
recent_layoff_round             int64
team_size   

In [33]:
print(f"duplicated = {raw_data.duplicated().sum()}")
print(f"size = {raw_data.size}")
print(f"shape = {raw_data.shape}\n")
print(f"Missing raw_data\n{raw_data.isnull().sum()}")

duplicated = 0
size = 1305000
shape = (45000, 29)

Missing raw_data
employee_id                   0
age                           0
generation                    0
gender                        0
department                    0
job_level                     0
is_manager                    0
tenure_years                  0
salary_usd                    0
salary_vs_market_pct          0
work_arrangement              0
rto_mandate                   0
days_in_office_required       0
commute_minutes               0
prefers_remote                0
flexibility_importance        0
ai_tools_adoption             0
weekly_hours                  0
after_hours_work              0
burnout_score                 0
manager_relationship_score    0
manager_1on1_per_month        0
engagement_score              0
promotions_last_2y            0
career_growth_score           0
recent_layoff_round           0
team_size                     0
attrition_reason              0
attrition                     0
dtyp

# cleaning

ตัด employee_id ไม่ได้ใช้

ตัด attrition_reason เหตุผลการลาออกไม่น่าได้ใช้คำนวณ

ตัด is_manager — ซ้ำ job_level

ตัด generation — กับ age เป็น Multicollinearity 

ก่อนตัด
45,000 แถว, 29 คอลัมน์

In [34]:
raw_data = raw_data.drop(columns=["employee_id", "attrition_reason", "is_manager","generation"])

ตัด เริ่มงานตั้งแต่อายุต่ำกว่า 16 เพราะว่ามีคนที่เริ่มงานตั้งแต่อายุ age=20, แต่อายุงาน tenure_years=9.8

In [35]:
bad_mask = raw_data["tenure_years"] > (raw_data["age"] - 16)
n_bad = bad_mask.sum()

clean_data = raw_data[~bad_mask].reset_index(drop=True)


In [36]:
clean_data.head()

,age,gender,department,job_level,tenure_years,salary_usd,salary_vs_market_pct,work_arrangement,rto_mandate,days_in_office_required,...,after_hours_work,burnout_score,manager_relationship_score,manager_1on1_per_month,engagement_score,promotions_last_2y,career_growth_score,recent_layoff_round,team_size,attrition
0,26,Female,Marketing,Junior,5.5,70000,5.2,onsite,0,4,...,0,3.6,2,0,62,0,2,0,22,0
1,35,Female,Sales,Junior,0.5,64000,-15.5,onsite,0,5,...,0,4.8,3,2,79,1,2,0,24,0
2,23,Male,HR,Senior,2.0,142000,-15.9,hybrid,1,3,...,0,10.0,1,2,43,0,4,1,3,1
3,20,Female,Sales,Manager,2.2,147000,2.6,onsite,0,5,...,1,4.3,1,4,64,1,3,0,24,0
4,40,Female,Data,Senior,6.0,141000,11.3,hybrid,1,4,...,0,2.9,1,1,92,0,2,0,5,0


# Outlier 

In [37]:
# 1. เช็คสถิติ IQR — เฉพาะคอล continuous จริง (สเกลตายตัวเช่น burnout 1-10 ไม่เช็คแบบนี้)
def iqr_check(s):
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((s < lo) | (s > hi)).sum()
    return lo, hi, n

for col in ["age", "tenure_years", "salary_usd", "commute_minutes", "weekly_hours"]:
    lo, hi, n = iqr_check(clean_data[col])
    print(f"{col:16s} ขอบ=({lo:.1f}, {hi:.1f})  นอกขอบ={n}  max จริง={clean_data[col].max()}")
    # เหตุผล: ถ้า max จริงเกิน hi ไม่มาก และเป็นค่าที่เป็นไปได้ทางธุรกิจ (เช่นเงินเดือน Director สูง) = ไม่ใช่ error

# 2. เช็ค logic ขัดแย้งกันเอง (สำคัญกว่าสถิติ)
n_conflict = ((clean_data.work_arrangement == "remote") & (clean_data.days_in_office_required > 0)).sum()
print(f"\nremote แต่ต้องเข้าออฟฟิศ = {n_conflict} แถว")
print("เหตุผล: remote ควรมี days_in_office_required = 0 เสมอ ถ้าไม่ใช่ = ข้อมูลขัดแย้งกันเอง แน่นอนว่าเป็น error")

# 3. เช็คว่าเงินเดือนสมเหตุผลไหมเทียบ job_level ตัวเอง (ไม่เทียบข้ามกลุ่ม)
print("\nเงินเดือนต่ำสุด-สูงสุด แยกตาม job_level:")
print(clean_data.groupby("job_level")["salary_usd"].agg(["min", "max"]))
print("เหตุผล: ถ้าเงินเดือน Junior สูงกว่า Director = ผิดปกติจริง, ถ้าเรียงตามลำดับปกติ = ไม่ใช่ error แค่กระจายกว้าง")

age              ขอบ=(12.5, 56.5)  นอกขอบ=281  max จริง=64
tenure_years     ขอบ=(-2.6, 8.9)  นอกขอบ=1461  max จริง=22.7
salary_usd       ขอบ=(-31500.0, 252500.0)  นอกขอบ=650  max จริง=345000
commute_minutes  ขอบ=(-29.5, 94.5)  นอกขอบ=1426  max จริง=180
weekly_hours     ขอบ=(23.0, 63.0)  นอกขอบ=75  max จริง=71

remote แต่ต้องเข้าออฟฟิศ = 8574 แถว
เหตุผล: remote ควรมี days_in_office_required = 0 เสมอ ถ้าไม่ใช่ = ข้อมูลขัดแย้งกันเอง แน่นอนว่าเป็น error

เงินเดือนต่ำสุด-สูงสุด แยกตาม job_level:
              min     max
job_level                
Director   118000  345000
Junior      35000   95000
Lead        80000  242000
Manager     88000  268000
Mid         44000  146000
Senior      60000  203000
เหตุผล: ถ้าเงินเดือน Junior สูงกว่า Director = ผิดปกติจริง, ถ้าเรียงตามลำดับปกติ = ไม่ใช่ error แค่กระจายกว้าง


In [38]:
# แก้เฉพาะที่พิสูจน์แล้วว่าขัดแย้งกันเอง (จาก cell B ข้อ 2)
clean_data.loc[clean_data["work_arrangement"] == "remote", "days_in_office_required"] = 0

# ค่าสุดขั้วที่เป็นไปได้จริง (จาก cell B ข้อ 1, 3) ไม่ลบแถว แค่ clip กันดึงโมเดลเพี้ยน
for col in ["salary_usd", "tenure_years", "commute_minutes"]:
    lo, hi = clean_data[col].quantile([.01, .99])
    clean_data[col] = clean_data[col].clip(lo, hi)

print("จัดการเสร็จ, shape:", clean_data.shape)

จัดการเสร็จ, shape: (43439, 25)


C:\Users\liket\AppData\Local\Temp\ipykernel_18316\2744028702.py:7: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clean_data[col] = clean_data[col].clip(lo, hi)


In [39]:
print("remote conflict เหลือ:",
      ((clean_data.work_arrangement == "remote") & (clean_data.days_in_office_required > 0)).sum())
print(clean_data[["salary_usd", "tenure_years", "commute_minutes"]].describe().loc[["min", "max"]])

remote conflict เหลือ: 0
     salary_usd  tenure_years  commute_minutes
min     47000.0           0.3              3.0
max    263620.0          11.4            118.0


# encoding 

job_level มีลำดับจริง (Junior<Mid<Senior<Lead<Manager<Director) OrdinalEncoder

ai_order มีลำดับจริง ("none<light<regular<power) OrdinalEncoder

gender, department, work_arrangement, ไม่มีลำดับ ใช้ OneHotEncoder


In [40]:
# OrdinalEncoder
job_level_order = ["Junior", "Mid", "Senior", "Lead", "Manager", "Director"]
clean_data["job_level"] = pd.Categorical(clean_data["job_level"], categories=job_level_order, ordered=True).codes
clean_data["job_level"].value_counts().sort_index()

job_level
0    10418
1    13001
2     9508
3     4452
4     4361
5     1699
Name: count, dtype: int64

In [41]:
# OrdinalEncoder
ai_order = ["none", "light", "regular", "power"]
clean_data["ai_tools_adoption"] = pd.Categorical(clean_data["ai_tools_adoption"], categories=ai_order, ordered=True).codes
clean_data["ai_tools_adoption"].value_counts().sort_index()

ai_tools_adoption
0     7846
1    14747
2    14779
3     6067
Name: count, dtype: int64

In [42]:
# OneHotEncoder
nominal_cols = ["gender", "department", "work_arrangement"]
clean_data = pd.get_dummies(clean_data, columns=nominal_cols ,drop_first=True)

In [43]:
clean_data

,age,job_level,tenure_years,salary_usd,salary_vs_market_pct,rto_mandate,days_in_office_required,commute_minutes,prefers_remote,flexibility_importance,...,department_Data,department_Engineering,department_Finance,department_HR,department_Marketing,department_Operations,department_Product,department_Sales,work_arrangement_onsite,work_arrangement_remote
0,26,0,5.5,70000,5.2,0,4,44,1,3,...,False,False,False,False,True,False,False,False,True,False
1,35,0,0.5,64000,-15.5,0,5,48,1,2,...,False,False,False,False,False,False,False,True,True,False
2,23,2,2.0,142000,-15.9,1,3,35,1,3,...,False,False,False,True,False,False,False,False,False,False
3,20,4,2.2,147000,2.6,0,5,16,0,5,...,False,False,False,False,False,False,False,True,True,False
4,40,2,6.0,141000,11.3,1,4,55,1,4,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43434,42,1,5.5,71000,-19.9,1,5,51,0,3,...,False,True,False,False,False,False,False,False,True,False
43435,46,4,7.2,216000,7.5,0,5,20,1,3,...,True,False,False,False,False,False,False,False,True,False
43436,30,1,1.6,114000,-18.5,0,2,41,0,1,...,False,True,False,False,False,False,False,False,False,False
43437,23,1,3.0,96000,-14.6,0,3,41,1,1,...,False,True,False,False,False,False,False,False,False,False


# สรุป
มีข้อมูล ทั้งหมด 43439 rows × 37 columns


